# Event Display Quicklook

This notebook reuses the event display logic from `mlpf/macros/draw_events.py` to plot a single event from a parquet file (hits + tracks) in 3D using Plotly.

In [1]:
import awkward as ak
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

def _is_sequence_array(arr):
    return (
        arr.ndim == 2
        or (arr.ndim == 1 and len(arr) > 0 and isinstance(arr[0], (list, np.ndarray)))
    )

def plot_event_display(parquet_path, event_index=0, show=True, save_png=True, png_filename=None):
    output = ak.from_parquet(parquet_path)
    if event_index < 0 or event_index >= len(output["X_hit"]):
        raise IndexError(f"event_index {event_index} out of range")

    symbols = ["cross", "x", "circle", "square", "diamond"]

    X_hit = output["X_hit"][event_index]
    X_track = output["X_track"][event_index]

    hit_x = np.array(X_hit[:, 6])
    hit_y = np.array(X_hit[:, 7])
    hit_z = np.array(X_hit[:, 8])
    hit_e = np.array(X_hit[:, 5])
    hit_type = np.array(X_hit[:, -2]) + 1
    genlink_hits = np.array(output["ygen_hit"][event_index])

    track_x = np.array(X_track[:, 12])
    track_y = np.array(X_track[:, 13])
    track_z = np.array(X_track[:, 14])
    genlink_tracks = np.array(output["ygen_track"][event_index])

    if len(hit_x) == 0 and len(track_x) == 0:
        print("No hits or tracks in this event.")
        return None

    if len(genlink_hits) > 0 and len(genlink_tracks) > 0:
        genlink_all = np.concatenate((genlink_hits, genlink_tracks), axis=0)
    elif len(genlink_hits) > 0:
        genlink_all = genlink_hits
    else:
        genlink_all = genlink_tracks

    cmin = genlink_all.min()
    cmax = genlink_all.max()

    fig = go.Figure()

    if len(hit_x) > 0:
        marker_sizes = 10 * hit_e / hit_e
        for tval in np.unique(hit_type):
            mask = hit_type == tval
            fig.add_trace(
                go.Scatter3d(
                    x=hit_x[mask],
                    y=hit_y[mask],
                    z=hit_z[mask],
                    mode="markers",
                    name=f"hit_type {tval}",
                    marker=dict(
                        size=marker_sizes[mask],
                        color=genlink_hits[mask],
                        colorscale="Turbo",
                        cmin=cmin,
                        cmax=cmax,
                        opacity=0.8,
                        symbol=symbols[int(tval) % len(symbols)],
                        colorbar=dict(title="genlink0"),
                        showscale=bool(tval == np.unique(hit_type)[0]),
                    ),
                    text=[
                        f"type={tt}, e={ee:.2f}, gen={gg}"
                        for tt, ee, gg in zip(hit_type[mask], hit_e[mask] * 1e3, genlink_hits[mask])
                    ],
                    hoverinfo="text",
                )
            )

    if len(track_x) > 0:
        if _is_sequence_array(track_x):
            for idx in range(len(track_x)):
                x = np.array(track_x[idx])
                y = np.array(track_y[idx])
                z = np.array(track_z[idx])
                fig.add_trace(
                    go.Scatter3d(
                        x=x,
                        y=y,
                        z=z,
                        mode="lines",
                        name="track",
                        line=dict(color="white", width=3),
                        showlegend=bool(idx == 0),
                    )
                )
        else:
            fig.add_trace(
                go.Scatter3d(
                    x=track_x,
                    y=track_y,
                    z=track_z,
                    mode="markers",
                    name="tracks",
                    marker=dict(
                        size=6,
                        color=genlink_tracks,
                        colorscale="Turbo",
                        cmin=cmin,
                        cmax=cmax,
                        opacity=0.9,
                        symbol="diamond",
                        showscale=False,
                    ),
                )
            )

    fig.update_layout(
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            xaxis=dict(
                backgroundcolor="black",
                gridcolor="gray",
                showbackground=True,
                zerolinecolor="gray",
            ),
            yaxis=dict(
                backgroundcolor="black",
                gridcolor="gray",
                showbackground=True,
                zerolinecolor="gray",
            ),
            zaxis=dict(
                backgroundcolor="black",
                gridcolor="gray",
                showbackground=True,
                zerolinecolor="gray",
            ),
        ),
        paper_bgcolor="black",
        plot_bgcolor="black",
        margin=dict(l=0, r=0, b=0, t=0),
        legend=dict(title="Hit Type"),
        title=f"Event {event_index}",
    )

    if save_png:
        if png_filename is None:
            png_filename = f"event_{event_index + 1:04d}.png"
        fig.write_image(png_filename)

    if show:
        fig.show()

    return fig

In [2]:
CLD_PARQUET_DIR = "/eos/user/v/vriecher/mlpf_arc/CLD/train/gun_ARC_10k/05"
ARC_PARQUET_DIR = "/eos/user/v/vriecher/mlpf_arc/CLD/train/gun_ARC_10k/arc"

EVENT_INDEX = 60
FILE_INDEX = 50

In [5]:
from arc_gen_extrapolation_check import (
    compute_residuals_over_events,
    plot_eta_phi_clusters_tracks,
    plot_event_3d,
    plot_residuals_compare,
 )

ARC_EVENT_INDEX = EVENT_INDEX

ARC_3D = plot_event_3d(
    ARC_PARQUET_DIR,
    event_index=ARC_EVENT_INDEX,
    file_index=FILE_INDEX,
    show=True,
    save_png=True,
    png_filename=f"event_arc_{ARC_EVENT_INDEX + 1:04d}_3d.png",
)

ARC_ETA_PHI = plot_eta_phi_clusters_tracks(
    ARC_PARQUET_DIR,
    event_index=ARC_EVENT_INDEX,
    file_index=FILE_INDEX,
    show=True,
    save_png=True,
    png_filename=f"event_arc_{ARC_EVENT_INDEX + 1:04d}_etaphi.png",
)

ARC_RESIDUALS = compute_residuals_over_events(ARC_PARQUET_DIR, max_events=10000)
CLD_RESIDUALS = compute_residuals_over_events(CLD_PARQUET_DIR, max_events=10000)

RESIDUALS_FIG = plot_residuals_compare(
    ARC_RESIDUALS,
    CLD_RESIDUALS,
    max_dr=0.02,
    bin_width=0.001,
    show=True,
    save_png=True,
    png_filename=f"residuals_arc_vs_cld_{ARC_EVENT_INDEX + 1:04d}.png",
)

In [4]:
import arc_gen_extrapolation_check as ag
print(ag.__file__)

/eos/home-v/vriecher/mlpf_arc/mlpf/notebooks/arc_gen_extrapolation_check.py
